In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import holidays as hd

In [ ]:
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

In [ ]:
weather_data = pd.read_csv('../data/raw/Weather_YVR.csv')

In [ ]:
# loading the datasets
residential_23 = pd.read_csv('../data/raw/Residential_23.csv') 
residential_24 = pd.read_csv('../data/raw/Residential_24.csv')
residential_25 = pd.read_csv('../data/raw/Residential_25.csv')
residential_27 = pd.read_csv('../data/raw/Residential_27.csv')
residential_28 = pd.read_csv('../data/raw/Residential_28.csv')

In [ ]:
print(residential_23.head(5))

In [ ]:
print(residential_24.head(5))

In [ ]:
print(residential_25.head(5))

In [ ]:
print(residential_27.head(5))

In [ ]:
print(residential_28.head(5))

In [ ]:
print(f"shape of 23 is: {residential_23.shape}")
print(f"shape of 24 is: {residential_24.shape}")
print(f"shape of 25 is: {residential_25.shape}")
print(f"shape of 27 is: {residential_27.shape}")
print(f"shape of 28 is: {residential_28.shape}")

In [ ]:
def datetime_parsing(df, column_name = 'date'):
    df[column_name] = pd.to_datetime(df[column_name])
    df = df[df[column_name].dt.year == 2019]
    return df


In [ ]:
residential_24 = datetime_parsing(residential_24)
residential_23 = datetime_parsing(residential_23)
residential_25 = datetime_parsing(residential_25)
residential_27 = datetime_parsing(residential_27)
residential_28 = datetime_parsing(residential_28)

In [ ]:
residential_27.head()

In [ ]:
residential_25.head()

In [ ]:
# new shape for year 2019

In [ ]:
print(f"shape of 23 is: {residential_23.shape}")
print(f"shape of 24 is: {residential_24.shape}")
print(f"shape of 25 is: {residential_25.shape}")
print(f"shape of 27 is: {residential_27.shape}")
print(f"shape of 28 is: {residential_28.shape}")

In [ ]:
print(residential_23.isnull().sum())
print(residential_24.isnull().sum())
print(residential_25.isnull().sum())
print(residential_27.isnull().sum())
print(residential_28.isnull().sum())

In [ ]:
residential_24 = residential_24.interpolate()
residential_23 = residential_23.interpolate()
residential_25 = residential_25.interpolate()
residential_27 = residential_27.interpolate()
residential_28 = residential_28.interpolate()

In [ ]:
print(residential_23.isnull().sum())
print(residential_24.isnull().sum())
print(residential_25.isnull().sum())
print(residential_27.isnull().sum())
print(residential_28.isnull().sum())

In [ ]:
residential_23 = residential_23.rename(columns={'energy_kWh': 'energy_23'})
residential_24 = residential_24.rename(columns={'energy_kWh': 'energy_24'})
residential_25 = residential_25.rename(columns={'energy_kWh': 'energy_25'})
residential_27 = residential_27.rename(columns={'energy_kWh': 'energy_27'})
residential_28 = residential_28.rename(columns={'energy_kWh': 'energy_28'})

In [ ]:
residential_23.columns

In [ ]:
df1 = pd.merge(residential_23, residential_24, on=['date', 'hour'], how='outer')

In [ ]:
df2 = pd.merge(df1, residential_25, on=['date', 'hour'], how='outer')

In [ ]:
df3 = pd.merge(df2, residential_27, on=['date', 'hour'], how='outer')

In [ ]:
df = pd.merge(df3, residential_28, on=['date', 'hour'], how='outer')

In [ ]:
df.head()

In [ ]:
df['avg_energy'] = df[['energy_23','energy_24','energy_25','energy_27','energy_28']].mean(axis=1)

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
24*365

In [ ]:
duplicates = df[df.duplicated(subset=['date', 'hour'], keep=False)]
print(duplicates.shape)
print(duplicates.sort_values(['date','hour']).head(20))

In [ ]:
print(duplicates['date'].unique())

In [ ]:
df = df.drop_duplicates(subset=['date', 'hour'], keep='first')

In [ ]:
# 1. Create the missing row with NaN energy values
missing_row = pd.DataFrame({
    'date': [pd.Timestamp('2019-03-10')],
    'hour': [2],
    'energy_23': [np.nan],
    'energy_24': [np.nan],
    'energy_25': [np.nan],
    'energy_27': [np.nan],
    'energy_28': [np.nan],
    'avg_energy': [np.nan]
})



In [ ]:
# concat the missing row
df = pd.concat([df, missing_row], ignore_index=True)

# sort so the new row lands in the right position
df = df.sort_values(['date', 'hour']).reset_index(drop=True)

# interpolate the NaN values
df = df.interpolate()

# verify
print(df.shape)  # should be (8760, 8)

In [ ]:
df.head()

In [ ]:
weather_data = datetime_parsing(weather_data)

In [ ]:
weather_data.head()

In [ ]:
weather_data.shape

In [ ]:
df = pd.merge(df, weather_data, on=['date', 'hour'], how='left')

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
print(df.isnull().sum())

In [ ]:
df = df.drop(columns=['weather', 'energy_23', 'energy_24', 
                       'energy_25', 'energy_27', 'energy_28'])

In [ ]:
df.head()

In [ ]:
df = df.interpolate()

In [ ]:
df.isnull().sum()

In [ ]:
df.head()

In [ ]:
df = df.bfill()

In [ ]:
df.head()

In [ ]:
df['is_weekend'] = df['date'].dt.dayofweek >= 5
df['is_weekend'] = df['is_weekend'].astype(int)

In [ ]:
bc_holidays = hd.Canada(prov='BC', years=2019)

In [ ]:
print(df.columns.tolist())
df['is_holiday'] = df['date'].apply(lambda x: 1 if x in bc_holidays else 0)
print(df[['date', 'hour', 'is_weekend', 'is_holiday']].head(10))

In [ ]:
print(df.shape)
print(df.columns.tolist())
print(df.isnull().sum())
print(df.head(3))

In [ ]:
df.to_csv('../data/processed/energy_features_2019.csv', index=False)

In [ ]:
plt.plot(df['avg_energy'])

In [ ]:
plt.scatter(df['temperature'], df['avg_energy'], alpha=0.1)

In [ ]:
sns.boxplot(x='is_weekend', y='avg_energy', data=df)

In [ ]:
hourly_avg = df.groupby('hour')['avg_energy'].mean()
plt.plot(hourly_avg.index, hourly_avg.values)